
# Interaction analysis for the top two study-adjusted CNMA components

This notebook extends the preliminary additive CNMA by focusing on the two components with the strongest favorable point estimates in the **study-adjusted** model:

1. `treatment_follow_up_frequency_every_(2-4 weeks)` — study-adjusted posterior mean ≈ **−1.375 PHQ-9 points**
2. `treatment_is_there_a_shared_medical_record` — study-adjusted posterior mean ≈ **−0.687 PHQ-9 points**

The analysis keeps **study-specific intercepts** and adds baseline patient characteristics.

It answers two related questions:

- **Component–component interaction:** is the effect of having both top components different from the sum of their separate effects?
- **Effect modification by baseline characteristics:** does the association of either top component with PHQ-9 change differ according to baseline patient characteristics?

The outcome follows the previous notebook:


$$
\Delta PHQ9_i = PHQ9_{followup,i} - PHQ9_{baseline,i}
$$


Negative values indicate improvement.

### Interaction effect size

For the two-component interaction, the raw interaction coefficient


$$
\beta_{12}
$$


is reported in **PHQ-9 points**. A negative value means the combination performs better than expected under simple additivity; a positive value means less benefit than expected under additivity.

A standardized interaction effect size is also reported as


$$
d_{interaction} = \frac{\beta_{12}}{\sigma}
$$


where 
$$
\sigma
$$
 is the residual standard deviation.



## 1. Setup and load enriched IPD


In [ ]:

from pathlib import Path

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
IPD_DATA_PATH = PROCESSED_DIR / "enriched_dataset.csv"

df = pd.read_csv(
    IPD_DATA_PATH,
    low_memory=False,
)

print("Rows:", len(df))
print("Columns:", df.shape[1])



## 2. Prespecify the top two components

These are selected from the **study-adjusted model**, not from the pooled sensitivity model.


In [ ]:
TOP_COMPONENTS = [
    "treatment_follow_up_frequency_every_(2-4 weeks)",
    "treatment_is_there_a_relapse_prevention_plan",
]

TOP_COMPONENT_LABELS = {
    TOP_COMPONENTS[0]: "Routine follow-up every 2–4 weeks",
    TOP_COMPONENTS[1]: "treatment_is_there_a_relapse_prevention_plan",
}

missing_top_components = [c for c in TOP_COMPONENTS if c not in df.columns]

print("Missing top-component columns:", missing_top_components)

assert not missing_top_components, (
    "One or both prespecified top components are missing from enriched_dataset.csv."
)



## 3. Identify randomized studies and construct PHQ-9 change

This reproduces the previous notebook's current outcome construction: baseline plus the **earliest available post-baseline PHQ-9** per patient.


In [ ]:

arm_counts = (
    df.groupby(["STUDY_ID", "study_arm"])
    .size()
    .unstack(fill_value=0)
)

ready_studies = arm_counts[
    (arm_counts.get("control", 0) > 0)
    & (arm_counts.get("intervention", 0) > 0)
].index.tolist()

df["phq9_total"] = pd.to_numeric(
    df["phq9_total"],
    errors="coerce",
)

df["follow_up_months"] = pd.to_numeric(
    df["follow_up_months"],
    errors="coerce",
)

baseline = (
    df[
        (df["STUDY_ID"].isin(ready_studies))
        & (df["follow_up_months"] == 0)
        & df["phq9_total"].notna()
        & df["study_arm"].isin(["control", "intervention"])
    ][
        ["STUDY_ID", "patient_id", "study_arm", "phq9_total"]
    ]
    .rename(columns={"phq9_total": "phq9_baseline"})
    .copy()
)

followup = (
    df[
        (df["STUDY_ID"].isin(ready_studies))
        & (df["follow_up_months"] > 0)
        & df["phq9_total"].notna()
        & df["study_arm"].isin(["control", "intervention"])
    ][
        ["STUDY_ID", "patient_id", "follow_up_months", "phq9_total"]
    ]
    .copy()
)

earliest_followup = (
    followup
    .sort_values("follow_up_months")
    .groupby(["STUDY_ID", "patient_id"], as_index=False)
    .first()
    .rename(columns={"phq9_total": "phq9_followup"})
)

phq_change = baseline.merge(
    earliest_followup,
    on=["STUDY_ID", "patient_id"],
    how="inner",
    validate="one_to_one",
)

phq_change["phq9_change"] = (
    phq_change["phq9_followup"]
    - phq_change["phq9_baseline"]
)

print("Patients:", len(phq_change))
print("Studies:", phq_change["STUDY_ID"].nunique())
print(
    phq_change.groupby(["STUDY_ID", "study_arm"]).size()
)



## 4. Attach the two component indicators


In [ ]:

patient_components = (
    df[
        ["STUDY_ID", "patient_id"] + TOP_COMPONENTS
    ]
    .drop_duplicates()
)

component_pattern_counts = (
    patient_components
    .groupby(["STUDY_ID", "patient_id"])
    .size()
)

assert component_pattern_counts.max() == 1, (
    "At least one patient has more than one top-component pattern."
)

analysis_df = phq_change.merge(
    patient_components,
    on=["STUDY_ID", "patient_id"],
    how="left",
    validate="one_to_one",
)

for col in TOP_COMPONENTS:
    analysis_df[col] = (
        analysis_df[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({
            "yes": 1,
            "no": 0,
            "in_person": 1,
            "by_phone": 1,
        })
    )

print("Missing component values:")
print(analysis_df[TOP_COMPONENTS].isna().sum())

assert analysis_df[TOP_COMPONENTS].isna().sum().sum() == 0



## 5. Attach baseline patient characteristics

`phq9_baseline` is always included because it is constructed above.

The enriched dataset may contain additional harmonized baseline characteristics under different column names. The block below searches a conservative set of common aliases for:

- age
- sex/gender
- baseline long-term condition/comorbidity indicator
- baseline antidepressant medication use

Only variables actually found in `enriched_dataset.csv` are added.

**Important:** inspect the printed matches before fitting the final model. If your project uses a different harmonized column name, add it to `BASELINE_ALIASES`.


In [ ]:

BASELINE_ALIASES = {
    "age": [
        "age",
        "age_baseline",
        "baseline_age",
        "age_0",
    ],
    "sex": [
        "sex",
        "gender",
        "sex_baseline",
        "gender_baseline",
    ],
    "ltc_baseline": [
        "ltc_0",
        "ltc_baseline",
        "long_term_condition_baseline",
        "long_term_condition",
        "comorbidity_baseline",
    ],
    "antidepressant_baseline": [
        "antidepressant_0",
        "antidepressant_baseline",
        "baseline_antidepressant",
        "antidepressant_use_baseline",
    ],
}

def first_existing_column(frame, candidates):
    for col in candidates:
        if col in frame.columns:
            return col
    return None

resolved_baseline_columns = {
    name: first_existing_column(df, aliases)
    for name, aliases in BASELINE_ALIASES.items()
}

resolved_baseline_columns = {
    name: col
    for name, col in resolved_baseline_columns.items()
    if col is not None
}

print("Resolved baseline columns:")
for name, col in resolved_baseline_columns.items():
    print(f"  {name}: {col}")

if not resolved_baseline_columns:
    print(
        "\nNo optional baseline columns were auto-resolved. "
        "The model can still run with baseline PHQ-9 only."
    )


In [ ]:
baseline_characteristic_columns = [
    resolved_baseline_columns["age"],
    resolved_baseline_columns["sex"],
]

baseline_characteristics = (
    df[
        df["follow_up_months"] == 0
    ][
        ["STUDY_ID", "patient_id"]
        + baseline_characteristic_columns
    ]
    .drop_duplicates()
)

baseline_characteristic_counts = (
    baseline_characteristics
    .groupby(["STUDY_ID", "patient_id"])
    .size()
)

print(
    "Patients with >1 distinct baseline age/sex row:",
    (baseline_characteristic_counts > 1).sum()
)

if baseline_characteristic_counts.max() > 1:
    raise ValueError(
        "At least one patient has conflicting age/sex values "
        "within baseline rows."
    )

analysis_df = analysis_df.merge(
    baseline_characteristics,
    on=["STUDY_ID", "patient_id"],
    how="left",
    validate="one_to_one",
)

print("Baseline age and sex added")
print(analysis_df["age"].describe())
print(analysis_df["sex"].value_counts(dropna=False))



## 6. Encode and standardize baseline characteristics

Continuous variables are standardized to mean 0 and SD 1 so their interaction coefficients represent the change in component effect for a **1-SD difference** in the baseline characteristic.

Binary/categorical variables are converted to indicator columns. The code deliberately prints all coding decisions for inspection.


In [ ]:

# Baseline PHQ-9 is included as an adjustment covariate
analysis_df["phq9_baseline_z"] = (
    analysis_df["phq9_baseline"]
    - analysis_df["phq9_baseline"].mean()
) / analysis_df["phq9_baseline"].std()

# Age: continuous, standardized
analysis_df["age"] = pd.to_numeric(
    analysis_df["age"],
    errors="coerce",
)

analysis_df["age_z"] = (
    analysis_df["age"]
    - analysis_df["age"].mean()
) / analysis_df["age"].std()

# Sex: Male = 0, Female = 1
analysis_df["sex_binary"] = analysis_df["sex"].map({
    "Male": 0,
    "Female": 1,
})

# Adjustment covariates
adjustment_covariates = [
    "phq9_baseline_z",
]

adjustment_labels = {
    "phq9_baseline_z": "Baseline PHQ-9 (1 SD)",
}

# Baseline characteristics used as effect modifiers
effect_modifier_covariates = [
    "age_z",
    "sex_binary",
]

effect_modifier_labels = {
    "age_z": "Age (1 SD)",
    "sex_binary": "Female vs Male",
}

print("Adjustment covariates:")
for col in adjustment_covariates:
    print(" ", col, "->", adjustment_labels[col])

print("\nEffect modifiers:")
for col in effect_modifier_covariates:
    print(" ", col, "->", effect_modifier_labels[col])

print("\nMissing values:")
print(
    analysis_df[
        [
            "phq9_baseline_z",
            "age_z",
            "sex_binary",
        ]
    ].isna().sum()
)


In [ ]:
missing_by_study = (
    analysis_df
    .groupby("STUDY_ID")
    .agg(
        n=("patient_id", "size"),
        age_missing=("age", lambda x: x.isna().sum()),
        sex_missing=("sex", lambda x: x.isna().sum()),
    )
)

missing_by_study["age_missing_pct"] = (
    100 * missing_by_study["age_missing"]
    / missing_by_study["n"]
)

missing_by_study["sex_missing_pct"] = (
    100 * missing_by_study["sex_missing"]
    / missing_by_study["n"]
)

missing_by_study

## 7. Build the complete-case interaction-analysis dataset

The interaction model requires observed values for every variable that enters the model.

For the combined age-and-sex analysis, the required variables are:

- study identifier
- PHQ-9 change outcome
- standardized baseline PHQ-9
- standardized age
- binary sex indicator
- the two selected treatment components

Rows with missing values in any of these fields are removed before fitting.

Because age is completely missing in `05_Bekelman_2015`, that study is excluded from the combined age-and-sex interaction model.

The modeled outcome remains:

$$
\Delta PHQ9_i
=
PHQ9_{followup,i}
-
PHQ9_{baseline,i}
$$

Negative values indicate improvement.

In [ ]:
required_cols = [
    "STUDY_ID",
    "phq9_change",
    "phq9_baseline_z",
    "age_z",
    "sex_binary",
] + TOP_COMPONENTS

model_df = (
    analysis_df[required_cols]
    .dropna()
    .copy()
)

print("Patients:", len(model_df))
print("Studies:", model_df["STUDY_ID"].nunique())

print("\nPatients by study:")
print(model_df.groupby("STUDY_ID").size())

In [ ]:

c1, c2 = TOP_COMPONENTS

support_table = (
    analysis_df
    .groupby(["STUDY_ID", c1, c2])
    .size()
    .rename("n")
    .reset_index()
)

print("Component-combination support by study:")
display(support_table)

print("\nOverall support:")
display(
    analysis_df
    .groupby([c1, c2])
    .size()
    .rename("n")
    .reset_index()
)


In [ ]:

required_cols = (
    ["STUDY_ID", "phq9_change"]
    + TOP_COMPONENTS
    + adjustment_covariates
    + effect_modifier_covariates
)

model_df = (
    analysis_df[required_cols]
    .dropna()
    .copy()
)

model_df["component_interaction"] = (
    model_df[c1] * model_df[c2]
)

print("Patients:", len(model_df))
print("Studies:", model_df["STUDY_ID"].nunique())

print("\nPatients by study:")
print(model_df.groupby("STUDY_ID").size())


## 8. Check support for the observed component combinations

Before fitting an interaction model, we need to verify which combinations of the two treatment components actually occur in the analysis sample.

Let:

$$
C_{1i}
$$

denote routine follow-up every 2–4 weeks, and:

$$
C_{2i}
$$

denote the presence of a relapse prevention plan.

The theoretically possible combinations are:

| Follow-up | Relapse prevention | Interpretation |
|---:|---:|---|
| 0 | 0 | Neither component |
| 1 | 0 | Follow-up only |
| 0 | 1 | Relapse prevention only |
| 1 | 1 | Both components |

The support table below shows which of these combinations are actually represented, both overall and within each study.

This step is important because interaction effects are difficult or impossible to identify when some component combinations never occur.

In [ ]:
c1, c2 = TOP_COMPONENTS

# Check support for the two top components
support_by_study = (
    model_df
    .groupby(["STUDY_ID", c1, c2])
    .size()
    .rename("n")
    .reset_index()
)

print("Component combinations by study:")
display(support_by_study)

print("\nOverall component combinations:")
overall_support = (
    model_df
    .groupby([c1, c2])
    .size()
    .rename("n")
    .reset_index()
)

display(overall_support)

## 9. Diagnose component-interaction identifiability

The explicit component-component interaction is:

$$
C_{1i}C_{2i}
$$

In the current six-study sample, relapse prevention occurs only when routine follow-up is also present.

Therefore, whenever:

$$
C_{2i}=1
$$

we also have:

$$
C_{1i}=1
$$

which implies:

$$
C_{1i}C_{2i}=C_{2i}
$$

for every patient.

The code below checks this directly and calculates the rank of the design matrix.

If the interaction column and the relapse-prevention column are identical, the design is rank deficient. A Bayesian model with informative priors can still be sampled, but the relapse-prevention main effect and the explicit component-component interaction cannot be cleanly separated by the likelihood.

In [ ]:
c1, c2 = TOP_COMPONENTS

model_df["component_interaction"] = (
    model_df[c1] * model_df[c2]
)

print("Are relapse prevention and the interaction identical?")
print(
    np.array_equal(
        model_df[c2].to_numpy(),
        model_df["component_interaction"].to_numpy(),
    )
)

print("\nDesign matrix rank:")

X_check = model_df[
    [
        c1,
        c2,
        "component_interaction",
        "age_z",
        "sex_binary",
    ]
].to_numpy(dtype=float)

print("Columns:", X_check.shape[1])
print("Rank:", np.linalg.matrix_rank(X_check))

## 10. Fit the study-adjusted Bayesian interaction model

We now fit the main exploratory interaction model.

The outcome is PHQ-9 change:

$$
\Delta PHQ9_i
$$

The model includes:

- study-specific intercepts
- main effects of the two selected components
- the explicit component-component interaction
- baseline PHQ-9 adjustment
- main effects of age and sex
- component-by-age interactions
- component-by-sex interactions

The model can be written as:

$$
\Delta PHQ9_i
=
\alpha_{study(i)}
+
\beta_1 C_{1i}
+
\beta_2 C_{2i}
+
\beta_{12}C_{1i}C_{2i}
+
\gamma PHQ9_{0,i}^{*}
+
\theta_A Age_i^{*}
+
\theta_S Sex_i
+
\delta_{1A}C_{1i}Age_i^{*}
+
\delta_{2A}C_{2i}Age_i^{*}
+
\delta_{1S}C_{1i}Sex_i
+
\delta_{2S}C_{2i}Sex_i
+
\epsilon_i
$$

where:

$$
\alpha_{study(i)}
$$

accounts for study-level differences.

The component main effects are:

$$
\beta_1
$$

and:

$$
\beta_2
$$

The explicit component-component interaction is:

$$
\beta_{12}
$$

The age effect-modification terms are:

$$
\delta_{1A}
$$

and:

$$
\delta_{2A}
$$

The sex effect-modification terms are:

$$
\delta_{1S}
$$

and:

$$
\delta_{2S}
$$

Regularizing priors are used because the current component structure is sparse and partly rank deficient.

In [ ]:
# --------------------------------------------------
# Study-adjusted Bayesian interaction model
# --------------------------------------------------

study_codes, study_names = pd.factorize(model_df["STUDY_ID"])

# Main component variables
X_components = model_df[
    [
        c1,
        c2,
    ]
].to_numpy(dtype=float)

# Component-component interaction
X_component_interaction = (
    model_df["component_interaction"]
    .to_numpy(dtype=float)
)

# Baseline adjustment
X_phq9 = (
    model_df["phq9_baseline_z"]
    .to_numpy(dtype=float)
)

# Main baseline characteristics
X_age = (
    model_df["age_z"]
    .to_numpy(dtype=float)
)

X_sex = (
    model_df["sex_binary"]
    .to_numpy(dtype=float)
)

# Component × age interactions
c1_x_age = (
    model_df[c1]
    * model_df["age_z"]
).to_numpy(dtype=float)

c2_x_age = (
    model_df[c2]
    * model_df["age_z"]
).to_numpy(dtype=float)

# Component × sex interactions
c1_x_sex = (
    model_df[c1]
    * model_df["sex_binary"]
).to_numpy(dtype=float)

c2_x_sex = (
    model_df[c2]
    * model_df["sex_binary"]
).to_numpy(dtype=float)

# Outcome
y = model_df["phq9_change"].to_numpy(dtype=float)

coords = {
    "study": study_names.tolist(),
    "component": [
        "follow_up_2_4_weeks",
        "relapse_prevention_plan",
    ],
    "observation": np.arange(len(model_df)),
}

with pm.Model(coords=coords) as interaction_model:

    # Study-specific intercepts
    alpha_study = pm.Normal(
        "alpha_study",
        mu=0,
        sigma=5,
        dims="study",
    )

    # Main effects of the two components
    beta_component = pm.Normal(
        "beta_component",
        mu=0,
        sigma=1,
        dims="component",
    )

    # Explicit component-component interaction
    beta_component_interaction = pm.Normal(
        "beta_component_interaction",
        mu=0,
        sigma=1,
    )

    # Baseline PHQ-9 adjustment
    gamma_phq9 = pm.Normal(
        "gamma_phq9",
        mu=0,
        sigma=1,
    )

    # Main age and sex effects
    theta_age = pm.Normal(
        "theta_age",
        mu=0,
        sigma=1,
    )

    theta_sex = pm.Normal(
        "theta_sex",
        mu=0,
        sigma=1,
    )

    # Component × age interactions
    delta_c1_age = pm.Normal(
        "delta_c1_age",
        mu=0,
        sigma=1,
    )

    delta_c2_age = pm.Normal(
        "delta_c2_age",
        mu=0,
        sigma=1,
    )

    # Component × sex interactions
    delta_c1_sex = pm.Normal(
        "delta_c1_sex",
        mu=0,
        sigma=1,
    )

    delta_c2_sex = pm.Normal(
        "delta_c2_sex",
        mu=0,
        sigma=1,
    )

    mu = (
        alpha_study[study_codes]
        + beta_component[0] * X_components[:, 0]
        + beta_component[1] * X_components[:, 1]
        + beta_component_interaction * X_component_interaction
        + gamma_phq9 * X_phq9
        + theta_age * X_age
        + theta_sex * X_sex
        + delta_c1_age * c1_x_age
        + delta_c2_age * c2_x_age
        + delta_c1_sex * c1_x_sex
        + delta_c2_sex * c2_x_sex
    )

    sigma = pm.HalfNormal(
        "sigma",
        sigma=5,
    )

    y_obs = pm.Normal(
        "y_obs",
        mu=mu,
        sigma=sigma,
        observed=y,
        dims="observation",
    )

    trace_interaction = pm.sample(
        draws=4000,
        tune=4000,
        chains=4,
        target_accept=0.99,
        random_seed=42,
    )

## 11. Summarize posterior estimates and convergence diagnostics

The table below reports posterior means, posterior standard deviations, 95% highest-density intervals, effective sample sizes, and convergence diagnostics.

For each parameter, the 95% HDI represents the range containing 95% of the posterior probability mass.

A well-converged model should generally show:

$$
\hat{R} \approx 1
$$

with reasonably large effective sample sizes.

For interpretation:

- negative PHQ-9 coefficients indicate more improvement
- positive PHQ-9 coefficients indicate less improvement
- interaction terms describe how the effect of a component changes with age or sex

Because the relapse-prevention main-effect column and the explicit component-interaction column are identical in this dataset, their individual coefficients should be interpreted cautiously even if the sampler converges.

In [ ]:
az.summary(
    trace_interaction,
    var_names=[
        "beta_component",
        "beta_component_interaction",
        "gamma_phq9",
        "theta_age",
        "theta_sex",
        "delta_c1_age",
        "delta_c2_age",
        "delta_c1_sex",
        "delta_c2_sex",
        "sigma",
    ],
    hdi_prob=0.95,
)

## 12. Forest plot of component and effect-modification estimates

The forest plot summarizes the posterior mean and 95% credible interval for the main component effects and interaction terms.

The vertical reference line at:

$$
0
$$

represents no association with PHQ-9 change.

Estimates entirely below zero favor greater improvement, while estimates entirely above zero favor less improvement.

Intervals that cross zero indicate substantial posterior uncertainty about the direction of the effect.

In [ ]:
az.plot_forest(
    trace_interaction,
    var_names=[
        "beta_component",
        "beta_component_interaction",
        "theta_age",
        "theta_sex",
        "delta_c1_age",
        "delta_c2_age",
        "delta_c1_sex",
        "delta_c2_sex",
    ],
    combined=True,
    hdi_prob=0.95,
)

plt.axvline(0, linestyle="--")
plt.title("Study-adjusted component and effect-modification estimates")
plt.show()

## 13. Illustrative patient-level prediction

To make the model easier to interpret, we next apply it to one hypothetical patient profile.

For illustration, we choose a patient with:

- age = 60 years
- sex = Female
- baseline PHQ-9 = 15
- study context = `08_Coventry_2015`

Age and baseline PHQ-9 must first be standardized using the same means and standard deviations used when fitting the model.

For age:

$$
Age^{*}
=
\frac{Age-\overline{Age}}
{s_{Age}}
$$

For baseline PHQ-9:

$$
PHQ9_{0}^{*}
=
\frac{PHQ9_0-\overline{PHQ9_0}}
{s_{PHQ9_0}}
$$

This patient profile is illustrative only; it is not intended to represent an actual participant.

In [ ]:
# --------------------------------------------------
# Illustrative patient
# --------------------------------------------------

patient_age = 60
patient_sex = 1          # Female = 1
patient_phq9 = 15
patient_study = "08_Coventry_2015"

# Standardize using the SAME means/SDs used for model fitting
patient_age_z = (
    patient_age - analysis_df["age"].mean()
) / analysis_df["age"].std()

patient_phq9_z = (
    patient_phq9 - analysis_df["phq9_baseline"].mean()
) / analysis_df["phq9_baseline"].std()

print("Standardized age:", patient_age_z)
print("Standardized baseline PHQ-9:", patient_phq9_z)

In [ ]:
posterior = trace_interaction.posterior

# Study-specific intercept
alpha = (
    posterior["alpha_study"]
    .sel(study=patient_study)
    .values
    .reshape(-1)
)

# Component effects
b1 = (
    posterior["beta_component"]
    .sel(component="follow_up_2_4_weeks")
    .values
    .reshape(-1)
)

b2 = (
    posterior["beta_component"]
    .sel(component="relapse_prevention_plan")
    .values
    .reshape(-1)
)

b12 = (
    posterior["beta_component_interaction"]
    .values
    .reshape(-1)
)

# Adjustment/main effects
gamma = posterior["gamma_phq9"].values.reshape(-1)
theta_age = posterior["theta_age"].values.reshape(-1)
theta_sex = posterior["theta_sex"].values.reshape(-1)

# Effect modification
d1_age = posterior["delta_c1_age"].values.reshape(-1)
d2_age = posterior["delta_c2_age"].values.reshape(-1)

d1_sex = posterior["delta_c1_sex"].values.reshape(-1)
d2_sex = posterior["delta_c2_sex"].values.reshape(-1)

## 14. Compute expected PHQ-9 change under each observed treatment scenario

We calculate the posterior expected PHQ-9 change for the illustrative patient under the component combinations that are actually observed in the data.

The three observed scenarios are:

### Neither component

$$
C_1=0,\qquad C_2=0
$$

### Routine follow-up only

$$
C_1=1,\qquad C_2=0
$$

### Routine follow-up plus relapse prevention

$$
C_1=1,\qquad C_2=1
$$

The prediction function evaluates the full posterior linear predictor for each scenario while keeping the patient's age, sex, baseline PHQ-9, and study context fixed.

In [ ]:
def predict_change(c1_value, c2_value):

    interaction = c1_value * c2_value

    mu = (
        alpha
        + b1 * c1_value
        + b2 * c2_value
        + b12 * interaction
        + gamma * patient_phq9_z
        + theta_age * patient_age_z
        + theta_sex * patient_sex
        + d1_age * c1_value * patient_age_z
        + d2_age * c2_value * patient_age_z
        + d1_sex * c1_value * patient_sex
        + d2_sex * c2_value * patient_sex
    )

    return mu

In [ ]:
pred_neither = predict_change(0, 0)
pred_followup = predict_change(1, 0)
pred_both = predict_change(1, 1)

## 15. Summarize predicted PHQ-9 change

For each treatment scenario, we summarize the posterior distribution of the expected PHQ-9 change.

The table reports:

- posterior mean
- lower bound of the 95% interval
- upper bound of the 95% interval

Because:

$$
\Delta PHQ9 < 0
$$

indicates improvement, the most negative predicted value corresponds to the greatest expected improvement.

These are posterior expectations for the specified patient profile, not observed outcomes.

In [ ]:
def summarize_prediction(samples):
    return {
        "mean": np.mean(samples),
        "hdi_2.5%": np.quantile(samples, 0.025),
        "hdi_97.5%": np.quantile(samples, 0.975),
    }


prediction_table = pd.DataFrame(
    {
        "Neither component": summarize_prediction(pred_neither),
        "Follow-up only": summarize_prediction(pred_followup),
        "Follow-up + relapse prevention": summarize_prediction(pred_both),
    }
).T

prediction_table

## 16. Compare treatment scenarios using posterior contrasts

Comparing separate prediction intervals is not sufficient to determine whether two treatment scenarios differ.

Instead, we directly calculate the posterior distribution of the pairwise contrast.

For example:

$$
\Delta_{\mathrm{both-followup}}
=
\widehat{\Delta PHQ9}_{\mathrm{both}}
-
\widehat{\Delta PHQ9}_{\mathrm{followup}}
$$

A negative contrast means the first scenario is predicted to produce greater improvement.

We calculate:

$$
\Delta_{\mathrm{both-followup}}
$$

$$
\Delta_{\mathrm{both-neither}}
$$

and:

$$
\Delta_{\mathrm{followup-neither}}
$$

If the 95% credible interval of a contrast contains zero, the posterior remains uncertain about which treatment scenario is better for that patient profile.

In [ ]:
contrast_both_vs_followup = (
    pred_both - pred_followup
)

contrast_both_vs_neither = (
    pred_both - pred_neither
)

contrast_followup_vs_neither = (
    pred_followup - pred_neither
)

contrast_table = pd.DataFrame({
    "Both vs follow-up only": summarize_prediction(
        contrast_both_vs_followup
    ),
    "Both vs neither": summarize_prediction(
        contrast_both_vs_neither
    ),
    "Follow-up only vs neither": summarize_prediction(
        contrast_followup_vs_neither
    ),
}).T

contrast_table